In [ ]:
# 이미지와 라벨 데이터 셋 이름 정렬

In [1]:
import re, shutil
from pathlib import Path

# ───────── 사용자 설정 ─────────
IMAGES_DIR   = Path("C:/Users/User/Downloads/midnight/images")        # 원본 이미지 폴더
LABELS_DIR   = Path("C:/Users/User/Downloads/midnight/labels")        # 원본 라벨 폴더
NEW_IMG_DIR  = Path("C:/Users/User/Downloads/midnight/new_images")    # 새 이미지 폴더 (미리 생성돼 있음)
NEW_LBL_DIR  = Path("C:/Users/User/Downloads/midnight/new_labels")    # 새 라벨 폴더
PREFIX       = "IMG_"                # 접두사(빈 문자열 "" 로 바꾸면 0001.jpg 형태)
PAD_WIDTH    = 4                     # 0001: 4자리, 001:3, … 
COPY_FILES   = True                  # True=복사, False=이동
# ────────────────────────────

PAT_IMG  = re.compile(rf"^{PREFIX}?(\d+)\.(jpg|jpeg|png)$", re.I)
PAT_JSON = re.compile(rf"^{PREFIX}?(\d+)\.json$", re.I)

# 1) 이미지 & 라벨 번호 매핑 ----------------------------------------------------------------
img_map  = {int(m.group(1)): p for p in IMAGES_DIR.iterdir() if (m := PAT_IMG.match(p.name))}
json_map = {int(m.group(1)): p for p in LABELS_DIR.iterdir() if (m := PAT_JSON.match(p.name))}

common_ids = sorted(img_map.keys() & json_map.keys())
missing    = (img_map.keys() ^ json_map.keys())
if missing:
    raise RuntimeError(f"짝이 없는 번호가 있습니다: {sorted(missing)}")

# 2) 새 번호 매핑 (가장 작은 번호 → 1)
new_id_map = {old_id: idx + 1 for idx, old_id in enumerate(common_ids)}

# 3) 파일 복사/이동 -------------------------------------------------------------------------
NEW_IMG_DIR.mkdir(exist_ok=True, parents=True)
NEW_LBL_DIR.mkdir(exist_ok=True, parents=True)

for old_id in common_ids:
    new_id   = new_id_map[old_id]
    new_stub = f"{PREFIX}{new_id:0{PAD_WIDTH}d}"

    # 이미지
    src_img = img_map[old_id]
    dst_img = NEW_IMG_DIR / f"{new_stub}{src_img.suffix.lower()}"
    (shutil.copy if COPY_FILES else shutil.move)(src_img, dst_img)

    # 라벨
    src_json = json_map[old_id]
    dst_json = NEW_LBL_DIR / f"{new_stub}.json"
    (shutil.copy if COPY_FILES else shutil.move)(src_json, dst_json)

print(f"✔ 완료: {len(common_ids)} 쌍이 {new_stub[:-PAD_WIDTH]}0001~ 순으로 정렬되었습니다.")


✔ 완료: 251 쌍이 IMG_0001~ 순으로 정렬되었습니다.
